# Basic C2C algorithm

This notebook demonstrates a basic standard cloud-to-cloud (C2C) distance workflow in `py4dgeo`.

**Implemented by**  
Ronald Tabernig ([\@tabernig](https://github.com/tabernig))

First, import the required packages and helpers:

In [ ]:
import py4dgeo
import numpy as np
import pooch

Load a small pair of demo point clouds from the same test data used in the other tutorials:

In [ ]:
# Set up pooch to download data from Zenodo
p = pooch.Pooch(base_url="doi:10.5281/zenodo.18432391/", path=pooch.os_cache("py4dgeo"))
p.load_registry_from_doi()

try:
    # Download and extract the dataset
    p.fetch("trier_sim.zip", processor=pooch.Unzip(members=["trier_sim"]))

    # Define path to the extracted data
    data_path = p.path / "trier_sim.zip.unzip"
    print(f"Data path: {data_path}")

    before_rockfall_file = (
        data_path / "trier_sim_epoch_0.laz"
    )  # Synthetic data of terrain before a simulated rockfall at Trier study site
    after_rockfall_file = (
        data_path / "trier_sim_epoch_1.laz"
    )  # Synthetic data of terrain after a simulated rockfall at Trier study site

    epoch1, epoch2 = py4dgeo.read_from_las(before_rockfall_file, after_rockfall_file)


except Exception as e:
    print(f"Failed to download or extract data: {e}")

Instantiate the minimal `C2C` method and run it on a small corepoint subset:

In [ ]:
corepoints = epoch1.cloud[::25]

c2c = py4dgeo.C2C(
    epochs=(epoch1, epoch2),
    corepoints=corepoints,
    max_distance=3.0,
)

distances = c2c.run()

In [ ]:
distances[:10], np.nanmean(distances)

You can optionally enforce mutual matches using `correspondence_filter="mutual_nearest_neighbors"`. Non-mutual matches are kept in place and set to `np.nan`.

In [ ]:
c2c_mnn = py4dgeo.C2C(
    epochs=(epoch1, epoch2),
    corepoints=corepoints,
    max_distance=3.0,
    correspondence_filter="mutual_nearest_neighbors",
)

distances_mnn = c2c_mnn.run()
np.sum(np.isnan(distances_mnn)), distances_mnn[:10]

In [ ]:
outputfilepath = "c2c_results.las"

py4dgeo.write_c2c_results_to_las(
    outputfilepath,
    c2c,
    attribute_dict={"distances_C2C": distances,
                    "distances_C2C_MNN": distances_mnn},
)